In [7]:
import numpy as np
import xarray as xr

In [8]:
def compute_Psat_w(T):
    """
    Returns water liquid saturation pressure in Pascal.

    Parameters
    ----------
    T : Union[float, np.ndarray]
        Temperature in Kelvin.

    Returns
    -------
    Union[float, np.ndarray]
        H2O liquid saturation pressure in Pascal.
    """
    return 100.0 * np.exp( - 6096.9385 / T \
                            + 16.635794 \
                            - 0.02711193 * T \
                            + 1.673952E-5 * T * T \
                            + 2.433502 * np.log( T ) 
    )

def compute_Psat_i(T):
    """
    Returns water solid saturation pressure in Pascal.

    Parameters
    ----------
    T : Union[float, np.ndarray]
        Temperature in Kelvin.

    Returns
    -------
    Union[float, np.ndarray]
        H2O solid saturation pressure in Pascal.
    """
    return 100.0 * np.exp( - 6024.5282 / T \
                        + 24.7219 \
                        + 0.010613868 * T \
                        - 1.3198825E-5 * T * T \
                        - 0.49382577 * np.log( T ) )

In [9]:
test_num = 1
run_num = 2

# Define the file paths
input_file_path = '/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/BASE_APCEMM_met.nc'
output_file_template = '/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/test_{}/APCEMM_met_validation_{}.nc'

# Open the input NetCDF file
ds = xr.open_dataset(input_file_path)
ds

<xarray.Dataset>
Dimensions:                (altitude: 125, time: 24)
Coordinates:
  * altitude               (altitude) float64 0.0 0.1 0.2 0.3 ... 12.2 12.3 12.4
  * time                   (time) datetime64[ns] 2023-01-01 ... 2023-01-01T23...
    reference_time         datetime64[ns] ...
Data variables:
    shear                  (altitude, time) float64 ...
    stretch                (time) float64 ...
    pressure               (altitude) float64 ...
    temperature            (altitude, time) float64 ...
    w                      (altitude, time) float64 ...
    relative_humidity_ice  (altitude, time) float64 ...
Attributes:
    description:  APCEMM input dataset
    data_source:  artificial

In [10]:
# Make this applicable to PCE by updating with sampled values and saving input files
# Start with 10 training examples
mean_Y = 0
sigma_Y = 0.5
altitudes = 125
timesteps = 24

scaling = 15
scaled_mean = 120

# Sample fluctuations in RH values
RHi_sampled = np.round(np.random.normal(mean_Y, sigma_Y, size=timesteps) * scaling + scaled_mean, 2)
RHi_sampled_matrix = np.tile(RHi_sampled, (16, 1))

# Check if RH_sampled has any zero or negative values
if np.any(RHi_sampled_matrix <= 0):
    print("RHi_sampled contains zero or negative values.")
else:
    print("RHi_sampled does not contain any zero or negative values.")

    if RHi_sampled[0] < 117:
        print("Warning: The first value of RHi_sampled is less than 117.")

RHi_sampled does not contain any zero or negative values.


In [11]:
dims = ('altitude', 'time')

# Replace the 250 hPa row of ds with the sampled RH timeseries
ds['relative_humidity_ice'][90:106, :] = RHi_sampled_matrix
# ds['relative_humidity'][14, :] = RH_w

# # specify initial contrail environmental temperature
# ds['temperature'][14, :] = 218.0 #[K]

# ds['shear'] = (dims, 4.0*np.ones((altitudes, timesteps), dtype=float)) #[m/s/km] vertical wind shear
# ds['w'] = (dims, 0.0*np.ones((altitudes, timesteps), dtype=float)) #[m/s] vertical velocity

In [12]:
# Save the changes to new output files
output_file_path = output_file_template.format(test_num,run_num)
ds.to_netcdf(output_file_path)

ds.close()